In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Cài đặt thư viện
!pip install transformers datasets rouge-score accelerate sentencepiece underthesea -q

print("✅ Đã cài đặt xong!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã cài đặt xong!


2. Import và cấu hình đường dẫn

In [ ]:
import json
import torch
import os
import numpy as np
from transformers import (
    MBartForConditionalGeneration,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
from underthesea import word_tokenize
from rouge_score import rouge_scorer

# === CẤU HÌNH ĐƯỜNG DẪN ===
DATA_DIR = "/content/drive/MyDrive/DoAn/dataset/VSum-HB"  
TRAIN_PATH = os.path.join(DATA_DIR, "train.jsonl")
VALID_PATH = os.path.join(DATA_DIR, "valid.jsonl")

# Thư mục lưu model (cũng trong Drive để không mất khi restart)
OUTPUT_DIR = os.path.join(DATA_DIR, "bartpho-summarization-model")
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "final_model")

# Tạo thư mục lưu model ngay từ đầu
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print(f"📁 Train file: {TRAIN_PATH}")
print(f"📁 Valid file: {VALID_PATH}")
print(f"📁 Output dir: {OUTPUT_DIR}")
print(f"📁 Best model dir: {BEST_MODEL_DIR}")
print(f"📁 Final model dir: {FINAL_MODEL_DIR}")

# Kiểm tra file tồn tại
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Không tìm thấy {TRAIN_PATH}")
if not os.path.exists(VALID_PATH):
    print(f"⚠️ Không tìm thấy valid file, sẽ chỉ train trên train set")



📁 Train file: /content/drive/MyDrive/DoAn/dataset/VSum-HB/train.jsonl
📁 Valid file: /content/drive/MyDrive/DoAn/dataset/VSum-HB/valid.jsonl
📁 Output dir: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model
📁 Best model dir: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/best_model
📁 Final model dir: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/final_model


3. Đọc dữ liệu từ file

In [ ]:
def load_jsonl(file_path):
    """Đọc file JSONL và trả về list các dict"""
    data = []
    if not os.path.exists(file_path):
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line.strip())
                    # Dataset của bạn có cấu trúc: {"article": "...", "chosen": "...", "reject": "..."}
                    data.append({
                        "article": item["article"],
                        "summary": item["chosen"]  # Dùng chosen làm summary target
                    })
                except json.JSONDecodeError as e:
                    print(f"Lỗi dòng {line_num}: {e}")
                    continue

    print(f"✅ Đọc {len(data)} mẫu từ {file_path}")
    return data

# Đọc dữ liệu
print("\n📖 Đang đọc dữ liệu...")
train_data = load_jsonl(TRAIN_PATH)
valid_data = load_jsonl(VALID_PATH)

print(f"\n📊 Tổng số liệu:")
print(f"   Train: {len(train_data)} samples")
print(f"   Valid: {len(valid_data)} samples")


📖 Đang đọc dữ liệu...
✅ Đọc 4500 mẫu từ /content/drive/MyDrive/DoAn/dataset/VSum-HB/train.jsonl
✅ Đọc 500 mẫu từ /content/drive/MyDrive/DoAn/dataset/VSum-HB/valid.jsonl

📊 Tổng số liệu:
   Train: 4500 samples
   Valid: 500 samples


4. Word Segmentation (bắt buộc cho BARTpho-word)

In [ ]:
def word_segment_for_bartpho(text):
    """Word segmentation cho BARTpho-word"""
    try:
        # Underthesea trả về list các từ
        segmented_words = word_tokenize(text)
        # Nối bằng khoảng trắng (BARTpho-word nhận format này)
        return ' '.join(segmented_words)
    except Exception as e:
        print(f"Lỗi segment: {e}")
        return text  # Fallback

def preprocess_data(data, max_article_len=512, max_summary_len=128):
    """Tiền xử lý: segment và lọc độ dài"""
    processed = []
    total = len(data)

    print(f"🔄 Đang xử lý {total} mẫu...")

    for i, item in enumerate(data):
        if (i + 1) % 500 == 0:
            print(f"   Đã xử lý: {i+1}/{total}")

        # Word segmentation
        article_seg = word_segment_for_bartpho(item["article"])
        summary_seg = word_segment_for_bartpho(item["summary"])

        # Lọc bỏ mẫu quá dài
        if len(article_seg.split()) <= max_article_len and len(summary_seg.split()) <= max_summary_len:
            processed.append({
                "article": article_seg,
                "summary": summary_seg
            })

    print(f"✅ Sau lọc: {len(processed)}/{total} mẫu")
    return processed

# Hyperparameters cho preprocessing
MAX_ARTICLE_LEN = 512
MAX_SUMMARY_LEN = 128

# Tiền xử lý
print("\nĐang word segmentation train set...")
train_processed = preprocess_data(train_data, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN)

print("\nĐang word segmentation validation set...")
valid_processed = preprocess_data(valid_data, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN)


✂️ Đang word segmentation train set...
🔄 Đang xử lý 4500 mẫu...
   Đã xử lý: 500/4500
   Đã xử lý: 1000/4500
   Đã xử lý: 1500/4500
   Đã xử lý: 2000/4500
   Đã xử lý: 2500/4500
   Đã xử lý: 3000/4500
   Đã xử lý: 3500/4500
   Đã xử lý: 4000/4500
   Đã xử lý: 4500/4500
✅ Sau lọc: 2550/4500 mẫu

✂️ Đang word segmentation validation set...
🔄 Đang xử lý 500 mẫu...
   Đã xử lý: 500/500
✅ Sau lọc: 294/500 mẫu


5. Load model BARTpho-word

In [ ]:
def load_model_and_tokenizer():
    """Load BARTpho-word model"""
    print("\n🤖 Đang load BARTpho-word...")
    model_name = "vinai/bartpho-word"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MBartForConditionalGeneration.from_pretrained(model_name)

    # Kiểm tra special tokens
    print(f"   BOS token: {tokenizer.bos_token} (id: {tokenizer.bos_token_id})")
    print(f"   EOS token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
    print(f"   PAD token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

    # Đưa lên GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    print(f"✅ Model loaded on {device}")
    print(f"📊 Số tham số: {model.num_parameters():,}")

    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()


🤖 Đang load BARTpho-word...


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   BOS token: <s> (id: 0)
   EOS token: </s> (id: 2)
   PAD token: <pad> (id: 1)
✅ Model loaded on cuda
📊 Số tham số: 551,435,264


6. Tokenization function

In [ ]:
# === TOKENIZATION FUNCTION - FIXED VERSION ===
def tokenize_function(examples, tokenizer, max_input_length=512, max_output_length=128):
    """Tokenize inputs và labels cho BARTpho-word"""

    # Tokenize article
    model_inputs = tokenizer(
        examples["article"],
        max_length=max_input_length,
        truncation=True,
        padding=False,
    )

    # Tokenize summary - BARTpho dùng chung tokenizer
    labels = tokenizer(
        examples["summary"],
        max_length=max_output_length,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Chuyển sang HuggingFace Dataset format
print("\n📦 Chuyển đổi sang Dataset format...")
train_dataset = Dataset.from_list(train_processed)
valid_dataset = Dataset.from_list(valid_processed)

print(f"✅ Train dataset: {len(train_dataset)} samples")
print(f"✅ Valid dataset: {len(valid_dataset)} samples")

# Tokenize
print("\n🔤 Đang tokenize train set...")
train_tokenized = train_dataset.map(
    lambda x: tokenize_function(x, tokenizer, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN),
    batched=True,
    remove_columns=train_dataset.column_names
)

print("🔤 Đang tokenize validation set...")
valid_tokenized = valid_dataset.map(
    lambda x: tokenize_function(x, tokenizer, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN),
    batched=True,
    remove_columns=valid_dataset.column_names
)

print("✅ Tokenization hoàn tất!")

# Kiểm tra một mẫu đã tokenize
print("\n📊 Kiểm tra mẫu tokenized:")
sample = train_tokenized[0]
print(f"   Input ids length: {len(sample['input_ids'])}")
print(f"   Labels length: {len(sample['labels'])}")
print(f"   First 10 input ids: {sample['input_ids'][:10]}")
print(f"   First 10 labels: {sample['labels'][:10]}")


📦 Chuyển đổi sang Dataset format...
✅ Train dataset: 2550 samples
✅ Valid dataset: 294 samples

🔤 Đang tokenize train set...


Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

🔤 Đang tokenize validation set...


Map:   0%|          | 0/294 [00:00<?, ? examples/s]

✅ Tokenization hoàn tất!

📊 Kiểm tra mẫu tokenized:
   Input ids length: 278
   Labels length: 91
   First 10 input ids: [0, 2815, 3069, 3491, 15, 240, 108, 3183, 1099, 34]
   First 10 labels: [0, 3647, 308, 2815, 3069, 3491, 25, 302, 835, 31918]


7. Metrics ROUGE

In [ ]:
def compute_metrics(eval_pred, tokenizer):
    """Tính ROUGE scores"""
    predictions, labels = eval_pred

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Thay -100 bằng pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Tính ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

    for pred, label in zip(decoded_preds, decoded_labels):
        if pred and label:
            score = scorer.score(pred, label)
            scores['rouge1'].append(score['rouge1'].fmeasure)
            scores['rouge2'].append(score['rouge2'].fmeasure)
            scores['rougeL'].append(score['rougeL'].fmeasure)

    return {
        'rouge1': np.mean(scores['rouge1']) if scores['rouge1'] else 0,
        'rouge2': np.mean(scores['rouge2']) if scores['rouge2'] else 0,
        'rougeL': np.mean(scores['rougeL']) if scores['rougeL'] else 0
    }

8. Training configuration ( Hugging Face Transformers Trainer API)

In [ ]:
# === HYPERPARAMETERS TỐI ƯU CHO T4 16GB ===
BATCH_SIZE = 2                    
GRADIENT_ACCUMULATION = 2        
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5                    # 5 epoch cho 5000+ mẫu
WARMUP_RATIO = 0.1                # Warmup 10% steps
MAX_ARTICLE_LEN = 384             # Giảm nhẹ để tiết kiệm VRAM
MAX_SUMMARY_LEN = 96
SAVE_STEPS = 200                  

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Evaluation / save
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=SAVE_STEPS,

    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    # Training
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,

    # Generation
    predict_with_generate=True,
    generation_max_length=MAX_SUMMARY_LEN,
    generation_num_beams=4,

    # Optimization
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",
    push_to_hub=False,

    # Memory
    remove_unused_columns=True,
)

print(f"📋 Training configuration:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"   Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Save every: {SAVE_STEPS} steps")
print(f"   FP16: {training_args.fp16}")



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


📋 Training configuration:
   Batch size: 2
   Gradient accumulation: 2
   Effective batch size: 4
   Learning rate: 2e-05
   Epochs: 5
   Save every: 200 steps
   FP16: True


9. Trainer và bắt đầu train

In [ ]:
# === HÀM TÌM CHECKPOINT GẦN NHẤT ===
def get_latest_checkpoint(output_dir):
    checkpoints = []
    if not os.path.exists(output_dir):
        return None

    for name in os.listdir(output_dir):
        if name.startswith("checkpoint-"):
            try:
                step = int(name.split("-")[-1])
                checkpoints.append((step, os.path.join(output_dir, name)))
            except ValueError:
                pass

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]

# === KHỞI TẠO TRAINER ===
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized if len(valid_tokenized) > 0 else None,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, tokenizer),
)

# Tìm checkpoint cũ để resume nếu có
latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)
if latest_checkpoint:
    print(f" Tìm thấy checkpoint gần nhất: {latest_checkpoint}")
    print(" Sẽ resume training từ checkpoint này")
else:
    print("Không có checkpoint cũ, bắt đầu train mới")

# Bắt đầu training
print("BẮT ĐẦU FINE-TUNE...")
print("=" * 50)

train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
trainer.save_state()

# Lưu model tốt nhất sau khi train
print(f"Lưu BEST model vào {BEST_MODEL_DIR}")
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Lưu thêm bản final để demo/inference
print(f"💾 Lưu FINAL model vào {FINAL_MODEL_DIR}")

trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("✅ HOÀN THÀNH!")
print(f"📁 Best model: {BEST_MODEL_DIR}")
print(f"📁 Final model: {FINAL_MODEL_DIR}")
print(f"📁 Checkpoints nằm trong: {OUTPUT_DIR}")



Không có checkpoint cũ, bắt đầu train mới
BẮT ĐẦU FINE-TUNE...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
200,4.613766,1.689718,0.683492,0.411866,0.443025
400,3.839743,1.571108,0.681394,0.421835,0.449708
600,3.558299,1.472856,0.734028,0.452657,0.479682
800,2.928490,1.482532,0.729594,0.458793,0.485367
1000,2.505203,1.478800,0.732891,0.457316,0.483763
1200,2.342160,1.463273,0.745473,0.469963,0.494751
1400,1.958824,1.514007,0.746497,0.470863,0.492521
1600,1.712344,1.521147,0.740559,0.466775,0.485591
1800,1.703354,1.537840,0.750941,0.474821,0.496610
2000,1.310121,1.557776,0.743100,0.469569,0.491363


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (14 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Lưu FINAL model vào /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/final_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ HOÀN THÀNH!
📁 Best model: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/best_model
📁 Final model: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/final_model
📁 Checkpoints nằm trong: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model


10. Code kiểm tra sau khi train

In [ ]:
def load_model_for_inference(model_path):
    """Load model đã train để inference"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = MBartForConditionalGeneration.from_pretrained(model_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    return model, tokenizer

def summarize(text, model, tokenizer, max_length=128):
    """Tóm tắt văn bản"""
    segmented = word_segment_for_bartpho(text)

    inputs = tokenizer(
        segmented,
        max_length=MAX_ARTICLE_LEN,
        truncation=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            min_length=20,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(" KIỂM TRA MÔ HÌNH:")


model_path = BEST_MODEL_DIR if os.path.exists(BEST_MODEL_DIR) else FINAL_MODEL_DIR
print(f"📁 Model dùng để demo: {model_path}")

model_infer, tokenizer_infer = load_model_for_inference(model_path)

raw_test_article = valid_data[0]["article"] if valid_data else train_data[0]["article"]
original_summary = valid_data[0]["summary"] if valid_data else train_data[0]["summary"]

print(" Bài báo gốc:")
print(raw_test_article[:1000] + ("..." if len(raw_test_article) > 1000 else ""))

print(" Tóm tắt gốc:")
print(original_summary)

print("Tóm tắt của model:")
predicted = summarize(raw_test_article, model_infer, tokenizer_infer)
print(predicted)

if torch.cuda.is_available():
    torch.cuda.empty_cache()



 KIỂM TRA MÔ HÌNH:
📁 Model dùng để demo: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/best_model


Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


 Bài báo gốc:
Giới chức Thái Lan tại hiện trường vụ tự sát . Ông Jirawut Pongphan , 42 tuổi , ở tỉnh Chonburi , phía đông Thái Lan , hồi tháng 11/2017 tuyên bố trúng giải độc đắc trị giá gần 1,4 triệu USD sau khi mua 7 vé số có cùng dãy số với hy vọng tối đa hoá giải thưởng có thể nhận được , Mirror hôm qua đưa tin . Trong niềm vui sắp đổi đời , ông tổ chức tiệc tại nhà để ăn mừng cùng gia đình và bạn bè . Tuy nhiên , khi tỉnh dậy vào hôm sau , ông phát hiện toàn bộ 7 chiếc vé đã biến mất . Việc mất vé số trúng thưởng khiến Jirawut bị trầm cảm nặng . Ông không thể đi làm vì căng thẳng và buồn chán . Ngày 31/1 , ông dùng súng ngắn bắn vào đầu tự sát . Cảnh sát tìm thấy thi thể người đàn ông trong phòng ngủ cùng lá thư tuyệt mệnh . " Tôi rất tiếc , làm ơn đừng bắt nạt gia đình tôi , tôi thật sự đã trúng số " , lá thư viết . Tung tích những chiếc vé số của Jirawut chưa được xác định . Hiện cũng không rõ đã có người đến nhận giải độc đắc hay chưa . Vũ Phong   Giới chức Thái Lan tại hiện tr

Test thử

In [ ]:
def show_model_dir_info(model_dir, title):
    print(f"{'='*60}")
    print(f"{title}: {model_dir}")
    print(f"Tồn tại: {os.path.exists(model_dir)}")

    if not os.path.exists(model_dir):
        return

    files = os.listdir(model_dir)
    if not files:
        print("(Thư mục rỗng)")
        return

    print("Các file trong thư mục:")
    for f in files:
        file_path = os.path.join(model_dir, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / (1024 * 1024)
            print(f"   - {f} ({size:.2f} MB)")
        else:
            print(f"   - {f} (folder)")

    important_files = ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json"]
    has_weight = any(x in files for x in ["model.safetensors", "pytorch_model.bin"])

    print(f"Có file trọng số model: {'✅' if has_weight else '❌'}")
    for imp in important_files:
        print(f"{imp}: {'✅' if imp in files else '❌'}")

show_model_dir_info(BEST_MODEL_DIR, "BEST MODEL")
show_model_dir_info(FINAL_MODEL_DIR, "FINAL MODEL")

latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)
print(f" Checkpoint gần nhất: {latest_checkpoint if latest_checkpoint else 'Không có'}")

# === DEMO NHẬP VĂN BẢN BẤT KỲ ĐỂ TÓM TẮT ===
text_demo = """
Ai sinh ra trên cuộc đời đều có mẹ, vì mẹ chính là người đã cho ta cuộc sống này. Em cũng vậy, mẹ chính là người tuyệt vời nhất đối với em, mẹ đã cho em được sinh ra, được lớn lên và được sống trong vòng tay yêu thương của gia đình. Mẹ em năm nay đã ngoài ba mươi tuổi, công việc của mẹ là bán hoa quả ngoài chợ, công việc đó rất vất vả, buổi sáng mẹ dậy sớm để sắp hàng ra chợ, đến tối muộn mẹ mới dọn hàng về. Vì hay thức khuya dậy sớm nên trông dáng mẹ rất gầy, nước da sạm đi và xuất hiện vài nếp nhăn. Mỗi buổi được nghỉ học em đều ra chợ phụ mẹ trông hàng, em mong sao có thể giúp đỡ mẹ để mẹ được nghỉ ngơi nhiều hơn.

"""

print("DEMO TÓM TẮT VĂN BẢN TỰ NHẬP:")
print("Văn bản đầu vào:")
print(text_demo)
print("Tóm tắt sinh ra:")
print(summarize(text_demo, model_infer, tokenizer_infer))



BEST MODEL: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/best_model
Tồn tại: True
Các file trong thư mục:
   - config.json (0.00 MB)
   - generation_config.json (0.00 MB)
   - model.safetensors (2103.86 MB)
   - tokenizer_config.json (0.00 MB)
   - added_tokens.json (0.00 MB)
   - vocab.txt (0.85 MB)
   - bpe.codes (1.08 MB)
   - training_args.bin (0.01 MB)
Có file trọng số model: ✅
config.json: ✅
generation_config.json: ✅
tokenizer.json: ❌
tokenizer_config.json: ✅
FINAL MODEL: /content/drive/MyDrive/DoAn/dataset/VSum-HB/bartpho-summarization-model/final_model
Tồn tại: True
Các file trong thư mục:
   - config.json (0.00 MB)
   - generation_config.json (0.00 MB)
   - model.safetensors (2103.86 MB)
   - tokenizer_config.json (0.00 MB)
   - added_tokens.json (0.00 MB)
   - vocab.txt (0.85 MB)
   - bpe.codes (1.08 MB)
Có file trọng số model: ✅
config.json: ✅
generation_config.json: ✅
tokenizer.json: ❌
tokenizer_config.json: ✅
 Checkpoint gần nhất: /content/drive/